In [11]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

# warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.append(root_path)

In [12]:
from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=True)

In [13]:
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
import datetime
from decimal import Decimal
from controllers.directional_trading.pz_directional import PZDirectionalControllerConfig


# Controller configuration
connector_name = "binance_perpetual"
trading_pair = "WLD-USDT"
interval = "5m"
backtesting_resolution = "5m"

# Indicator Values
# ema_short: int = 20
ema_medium: int = 20
ema_slow: int = 50
hma_fast: int = 10
hma_slow: int = 20
srsi_length:int = 9
srsi_smoothing: int = 3
rsi_ma_length: int = 9
#####

total_amount_quote = 1000
max_executors_per_side = 2
time_limit = 60 * 60 * 12 * 9999 # disable time limit
cooldown_time = 60 * 15
take_profit = 1.0 # 100%, -> Disable Take profit, let the trailing do it's job
stop_loss = 0.01
trailing_stop_activation_price = 0.015
trailing_stop_trailing_delta = 0.005
sl_natr_factor = 1.0
natr_length = 14
ts_activation_natr_factor = 5.0
ts_delta_natr_factor = 5.5


# Creating the instance of the configuration and the controller
config = PZDirectionalControllerConfig(
    connector_name=connector_name,
    trading_pair=trading_pair,
    interval=interval,
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    trailing_stop=TrailingStop(activation_price=Decimal(trailing_stop_activation_price), trailing_delta=Decimal(trailing_stop_trailing_delta)),
    total_amount_quote=Decimal(total_amount_quote),
    time_limit=time_limit,
    max_executors_per_side=max_executors_per_side,
    cooldown_time=cooldown_time,
    hma_fast=hma_fast,
    hma_slow=hma_slow,
    srsi_length=srsi_length,
    srsi_smoothing=srsi_smoothing,
    rsi_ma_length=rsi_ma_length,
    natr_length = natr_length,
    sl_natr_factor=sl_natr_factor,
    ts_activation_natr_factor = ts_activation_natr_factor,
    ts_delta_natr_factor = ts_delta_natr_factor,
    # ema_short=ema_short,
    ema_medium=ema_medium,
    ema_slow=ema_slow,
)

In [14]:
# Running the backtesting this will output a backtesting result object that has built in methods to visualize the results

start = int(datetime.datetime(2025, 2, 1).timestamp())
end = int(datetime.datetime(2025, 2, 7).timestamp())

backtesting_result = await backtesting.run_backtesting(config, start, end, backtesting_resolution)

2025-04-06 15:28:54,440 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x769e06a16ad0>
2025-04-06 15:28:54,441 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x769e27bcfdc0>, 70075.418725655)])']
connector: <aiohttp.connector.TCPConnector object at 0x769e06a16b00>
2025-04-06 15:28:55,596 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x769e06a787f0>
2025-04-06 15:28:55,597 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x769e06a49240>, 70076.574807651)])']
connector: <aiohttp.connector.TCPConnector object at 0x769e06a78820>
2025-04-06 15:28:56,194 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x769e06a140d0>
2025-04-06 15:28:56,195 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.cli

In [15]:

# candles_df = backtesting_result.processed_data
# candles_df

In [16]:
import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# Let's see what is inside the backtesting results
print(backtesting_result.get_results_summary())
fig = backtesting_result.get_backtesting_figure()
# Add EMAs
candles_df = backtesting_result.processed_data



hma_fast_key = f"HMA_{hma_fast}"
hma_slow_key = f"HMA_{hma_slow}"
# ema_fast = f'EMA_{ema_short}'
ema_medium_key = f'EMA_{ema_medium}'
ema_slow_key = f'EMA_{ema_slow}'
k_key = [f"STOCHRSIk_{srsi_length}_{srsi_length}_{srsi_smoothing}_{srsi_smoothing}"]


fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[hma_fast_key],
                         line=dict(color='#00FF00', width=2),
                         name='Fast HMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[hma_slow_key],
                         line=dict(color='#FFA500', width=2),
                         name='Slow HMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_slow_key],
                         line=dict(color='#FFFFFF', width=2),
                         name='Slow EMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_medium_key],
                         line=dict(color='#FFFFFF', width=2),
                         name='Slow EMA'))



Net PNL: $44.82 (4.48%) | Max Drawdown: $-54.09 (-5.42%)
Total Volume ($): 73000.00 | Sharpe Ratio: 0.94 | Profit Factor: 1.30
Total Executors: 98 | Accuracy Long: 0.22 | Accuracy Short: 0.23
Close Types: Take Profit: 0 | Stop Loss: 48 | Time Limit: 0 |
             Trailing Stop: 0 | Early Stop: 50



In [17]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Grab processed data
candles_df = backtesting_result.processed_data

# Keys
hma_fast_key = f"HMA_{hma_fast}"
hma_slow_key = f"HMA_{hma_slow}"
ema_slow_key = f'EMA_{ema_slow}'
rsi_key = f'RSI_{rsi_ma_length}'
wma_key = f'WMA_{rsi_ma_length}'
k_key = f"STOCHRSIk_{srsi_length}_{srsi_length}_{srsi_smoothing}_{srsi_smoothing}"
d_key = f"STOCHRSId_{srsi_length}_{srsi_length}_{srsi_smoothing}_{srsi_smoothing}"

# Create subplots layout
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    row_heights=[0.45, 0.2, 0.2, 0.15],
    vertical_spacing=0.02,
    subplot_titles=["Price + Indicators", "RSI and WMA", "StochRSI %K & %D", "Signal"]
)

# Row 1: Close price
fig.add_trace(go.Scatter(
    x=candles_df.index, y=candles_df["close"],
    line=dict(color='#FFFFFF', width=2), name='Close'), row=1, col=1)


# Row 1: HMAs and EMA
fig.add_trace(go.Scatter(
    x=candles_df.index, y=candles_df[hma_fast_key],
    line=dict(color='#00FF00', width=2), name='Fast HMA'), row=1, col=1)

fig.add_trace(go.Scatter(
    x=candles_df.index, y=candles_df[hma_slow_key],
    line=dict(color='#FFA500', width=2), name='Slow HMA'), row=1, col=1)

# fig.add_trace(go.Scatter(
#     x=candles_df.index, y=candles_df[ema_slow_key],
#     line=dict(color='#FFFFFF', width=2), name='Slow EMA'), row=1, col=1)

# Row 2: RSI and WMA
fig.add_trace(go.Scatter(
    x=candles_df.index, y=candles_df[rsi_key],
    line=dict(color='#1E90FF', width=2), name='RSI'), row=2, col=1)

fig.add_trace(go.Scatter(
    x=candles_df.index, y=candles_df[wma_key],
    line=dict(color='#FFD700', width=2, dash='dash'), name='RSI WMA'), row=2, col=1)

# Row 3: StochRSI %K and %D
fig.add_trace(go.Scatter(
    x=candles_df.index, y=candles_df[k_key],
    line=dict(color='#00CED1', width=2), name='%K'), row=3, col=1)

fig.add_trace(go.Scatter(
    x=candles_df.index, y=candles_df[d_key],
    line=dict(color='#FF69B4', width=2, dash='dot'), name='%D'), row=3, col=1)

# Row 4: Signal values as line + markers
fig.add_trace(go.Scatter(
    x=candles_df.index,
    y=candles_df["signal"],
    mode='lines+markers',
    line=dict(color='cyan', width=2),
    name='Signal',
    marker=dict(size=6)
), row=4, col=1)

# Add optional 0-line
fig.add_shape(type="line",
              x0=candles_df.index[0],
              x1=candles_df.index[-1],
              y0=0, y1=0,
              line=dict(color="gray", dash="dot"),
              row=4, col=1)


# Final layout
fig.update_layout(
    height=700,
    title="Backtest with Candles, HMAs, RSI, and StochRSI",
    showlegend=True,
    template="plotly_dark"
)

fig.show()



In [18]:
# # 2. The executors dataframe: this is the dataframe that contains the information of the orders that were executed
import pandas as pd

executors_df = backtesting_result.executors_df
executors_df

,id,timestamp,type,close_timestamp,close_type,status,config,net_pnl_pct,net_pnl_quote,cum_fees_quote,filled_amount_quote,is_active,is_trading,custom_info,controller_id,side
0,9CgLZUAUD1eXn3UcN4871k83jGVRFM2eNzgreHFviqV1,1738385700,position_executor,1738386300,CloseType.STOP_LOSS,RunnableStatus.TERMINATED,{'id': '9CgLZUAUD1eXn3UcN4871k83jGVRFM2eNzgreH...,-0.0042537380550871743689622661577232065610587...,-2.1268690275435870873366184241604059934616088...,0.29999999999999993338661852249060757458209991...,999.9999999999998863131622783839702606201171875,False,False,"{'close_price': 1.7855, 'level_id': None, 'sid...",None,SELL
1,9fvAK2ZN3ioQKKtNZMd5J6zpM7c8uTo2KRMArAeBhM6C,1738387500,position_executor,1738387800,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': '9fvAK2ZN3ioQKKtNZMd5J6zpM7c8uTo2KRMArA...,-0.0018871453354972304861103538087263586930930...,-0.9435726677486153679552671746932901442050933...,0.29999999999999998889776975374843459576368331...,500.00000000000005684341886080801486968994140625,False,True,"{'close_price': 1.7846, 'level_id': None, 'sid...",None,BUY
2,EbwxA6Xu5DjDVeFC5MUUCkF1i9moT1jgATP7UCVRL94z,1738394700,position_executor,1738396800,CloseType.STOP_LOSS,RunnableStatus.TERMINATED,{'id': 'EbwxA6Xu5DjDVeFC5MUUCkF1i9moT1jgATP7UC...,-0.0005999999999998889251354161622487026761518...,-0.2999999999999444777465384959214134141802787...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 1.7611, 'level_id': None, 'sid...",None,SELL
3,AGxF9boAze6zfEs22q82iDzFQTUEnr1DEtAmz5qTHPzP,1738389900,position_executor,1738405200,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': 'AGxF9boAze6zfEs22q82iDzFQTUEnr1DEtAmz5...,0.03302663229277584544529844379212590865790843...,16.51331614638792188998195342719554901123046875,0.29999999999999998889776975374843459576368331...,500,False,True,"{'close_price': 1.7243, 'level_id': None, 'sid...",None,SELL
4,FxG3cTmDtNBrnLGNkw4TD6YpMjMdB4VuW3VRKKzJRPHP,1738398000,position_executor,1738405200,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': 'FxG3cTmDtNBrnLGNkw4TD6YpMjMdB4VuW3VRKK...,0.01307120466765792146657254590991215081885457...,6.53560233382896083043078760965727269649505615...,0.29999999999999998889776975374843459576368331...,500,False,True,"{'close_price': 1.7243, 'level_id': None, 'sid...",None,SELL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,3yaiux55VQzDuSFioYP6LpiHqM3YJUioNTz9UTNkeXKa,1738855800,position_executor,1738862700,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': '3yaiux55VQzDuSFioYP6LpiHqM3YJUioNTz9UT...,0.01676613603473258232168241477211267920210957...,8.38306801736629125798572204075753688812255859375,0.29999999999999998889776975374843459576368331...,500,False,True,"{'close_price': 1.2222, 'level_id': None, 'sid...",None,SELL
94,ECXLehc12CncCDTuPn4RmFw2MmqGyCrpAHGTCRKxcwZB,1738867800,position_executor,1738869000,CloseType.STOP_LOSS,RunnableStatus.TERMINATED,{'id': 'ECXLehc12CncCDTuPn4RmFw2MmqGyCrpAHGTCR...,-0.0068992125984253631354459201929785194806754...,-3.4496062992126814705784454417880624532699584...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 1.2141, 'level_id': None, 'sid...",None,SELL
95,GqxALr4avQJwGz7QHddf8bYGaoLfPCxJbQABsCSebA2x,1738864500,position_executor,1738869000,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': 'GqxALr4avQJwGz7QHddf8bYGaoLfPCxJbQABsC...,-0.0031598678777865034372762664816036703996360...,-1.5799339388932516214936185861006379127502441...,0.29999999999999998889776975374843459576368331...,500,False,True,"{'close_price': 1.2141, 'level_id': None, 'sid...",None,SELL
96,BMfYKy5Rb3dXui3xJmmR9TkrhcsQCPW2NoLCMEALa4q5,1738870800,position_executor,1738872000,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': 'BMfYKy5Rb3dXui3xJmmR9TkrhcsQCPW2NoLCME...,-0.0029163467902052035707338717429593089036643...,-1.4581733951026016882224212167784571647644042...,0.29999999999999998889776975374843459576368331...,500,False,True,"{'close_price': 1.2116, 'leve

### Backtesting Analysis

### Scatter of PNL per Trade
This bar chart illustrates the PNL for each individual trade. Positive PNLs are shown in green and negative PNLs in red, providing a clear view of profitable vs. unprofitable trades.


In [19]:
# import plotly.express as px

# # Create a new column for profitability
# executors_df['profitable'] = executors_df['net_pnl_quote'] > 0

# # Create the scatter plot
# fig = px.scatter(
#     executors_df,
#     x="timestamp",
#     y='net_pnl_quote',
#     title='PNL per Trade',
#     color='profitable',
#     color_discrete_map={True: 'green', False: 'red'},
#     labels={'timestamp': 'Timestamp', 'net_pnl_quote': 'Net PNL (Quote)'},
#     hover_data=['filled_amount_quote', 'side']
# )

# # Customize the layout
# fig.update_layout(
#     xaxis_title="Timestamp",
#     yaxis_title="Net PNL (Quote)",
#     legend_title="Profitable",
#     font=dict(size=12, color="white"),
#     showlegend=False,
#     plot_bgcolor='rgba(0,0,0,0.8)',  # Dark background
#     paper_bgcolor='rgba(0,0,0,0.8)',  # Dark background for the entire plot area
#     xaxis=dict(gridcolor="gray"),
#     yaxis=dict(gridcolor="gray")
# )

# # Add a horizontal line at y=0 to clearly separate profits and losses
# fig.add_hline(y=0, line_dash="dash", line_color="lightgray")

# # Show the plot
# fig.show()

### Histogram of PNL Distribution
The histogram displays the distribution of PNL values across all trades. It helps in understanding the frequency and range of profit and loss outcomes.


In [20]:
# fig = px.histogram(executors_df, x='net_pnl_quote', title='PNL Distribution')
# fig.show()


# Conclusion
We can see that the indicator has potential to bring good signals to trade and might be interesting to see how we can design a market maker that shifts the mid price based on this indicator.
A lot of the short signals are wrong but if we zoom in into the loss signals we can see that the losses are not that big and the wins are bigger and if we had implemented the trailing stop feature probably a lot of them are going to be profits.

# Next steps
- Filter only the loss signals and understand what you can do to prevent them
- Try different configuration values for the indicator
- Test in multiple markets, pick mature markets like BTC-USDT or ETH-USDT and also volatile markets like DOGE-USDT or SHIB-USDT